# Soldani - Qualitative reporting pipeline (FairMind -> LLM -> scoring)

This notebook closes the implementation. All quantitative work stays with FairMind: the LLM receives the five effects already computed and a rigid LaTeX template, and is responsible only for the qualitative interpretation and for answering the *Recap Questions*. A deterministic scorer then measures how consistent those answers are with the exact numbers.

It is the counterpart of `2_3_benchmark_thor.ipynb`, where the LLM *computed* the effects and its numerical error was measured. That comparison is a step of the investigation, not the final architecture: here the model never produces a number, and this is precisely what makes its output verifiable.

Pipeline: `run_fairmind()` -> `build_prompts()` -> `call_llm_report()` -> `score_report()`.

Everything that was corrected in `2_3` applies here as well, since the ground truth is computed by the same code: Laplace smoothing with alpha = 1, both forms of the indirect effect, and prompt caching disabled for reproducibility.

## 1. Initial setup

Locates the repository root by walking up from the current working directory until it finds the `src/` package, then adds it to `sys.path`. Identical to the corresponding cell in `2_3_benchmark_thor.ipynb`.

In [ ]:
from pathlib import Path
import sys

# Find the root by searching the "src" folder
current = Path.cwd()

while current != current.parent:
    if (current / "src").exists():
        REPO_ROOT = current
        break
    current = current.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

## 2. Imports and LLM client configuration

Besides the FairMind modules, imports the three components of the reporting pipeline in `src/report_pipeline/`: the prompt builder, the client that returns raw text, and the validator that assigns the score. The llama.cpp endpoint is read from `LLAMA_HOST`/`LLAMA_PORT` as in the other notebooks.

In [ ]:
import json
import os
import pandas as pd

from pgmpy.estimators import BayesianEstimator
from src.graph import build_sfm
from src.model import fit_discrete_bayesian_model
from src.effects import (
    total_variation, total_effect,
    natural_direct_effect, natural_indirect_effect,
)
from src.llm import LLM_CONFIGS

from src.report_pipeline.prompt_builder import build_prompts
from src.report_pipeline.llm_client import (
    call_llm_report, extract_latex_document, find_unfilled_placeholders,
)
from src.report_pipeline.validator import (
    score_report, GROUND_TRUTH_RULES,
    THRESH_DE, EPS_IE, THRESH_TV, THRESH_SE_REL,
)
from src.report_pipeline.annotate import annotate_recap_answers

LLAMA_HOST = os.environ.get("LLAMA_HOST", "localhost")
LLAMA_PORT = os.environ.get("LLAMA_PORT", "8080")
LLM_CONFIGS[0]["base_url"] = f"http://{LLAMA_HOST}:{LLAMA_PORT}/v1"

print(f"LLM endpoint configurato: http://{LLAMA_HOST}:{LLAMA_PORT}/v1")

## 3. Benchmark configuration

The same `CONFIG` as `2_3_benchmark_thor.ipynb`: Adult dataset, protected attribute `S2_gender` (Female to Male), target `T_income` (`>50K`), mediator `hours-per-week`, confounder `education`.

In [ ]:
CONFIG = {
    "dataset_name": "adult",
    "csv_path": "../../data/processed/adult.csv",
    "target_col":  "T_income",
    "target_val":  ">50K",
    "protected":   "S2_gender",
    "x0": "Female",
    "x1": "Male",
    "mediators":   ["hours-per-week"],
    "confounders": ["education"],
}

## 4. FairMind - the only quantitative source

Identical to the corresponding cell in `2_3_benchmark_thor.ipynb`: builds the SFM, fits the Bayesian Network with Laplace smoothing (pseudo-count alpha = 1), applies the binning of `hours-per-week` and `education`, and computes the effects by exact inference, with SE derived as `TV - TE` (Eq. 3).

The indirect effect is computed in **both** forms the paper defines. The distinction matters here more than in `2_3`: the report presents a decomposition, and the Recap Questions are scored against `IE_reverse`, the form for which `TE = DE - IE` closes. Using the direct form would flip the answer to Q2 without any visible symptom, so the cell prints the identity as a check.

**These numbers are the only quantitative input of the pipeline**: nothing downstream recomputes them.

In [ ]:
import time

def run_fairmind(config: dict) -> tuple[dict, "DiscreteBayesianNetwork", int, float]:
    df = pd.read_csv(config["csv_path"])
    cols = (
        [config["protected"]]
        + config["mediators"]
        + config["confounders"]
        + [config["target_col"]]
    )
    df = df[cols].dropna()

    # Binned here, in-place, once: the BN fitted on this binned data is the
    # SAME instance build_llm_prompt() queries to build the tables given to
    # the LLM (supervisor's Point 4), so both sides start from identical
    # numbers on every cell, including the sparsest ones.
    if "hours-per-week" in df.columns:
        df["hours-per-week"] = pd.cut(
            df["hours-per-week"],
            bins=[0, 20, 35, 45, 60, 100],
            labels=["<=20", "21-35", "36-45", "46-60", ">60"],
            include_lowest=True,
        )

    # education: 16 levels -> 5 tiers. With 16 levels * 5 hour bins = 80
    # (z,w) combinations Qwen2.5-14B could not finish DE/IE: the response was
    # truncated even at max_tokens=16384, and a compact output format made it
    # worse (it returned invented values, e.g. TE identical to TV). With 5
    # tiers the combinations drop to 25. The tiers follow the standard
    # grouping used in the Adult literature (cf. Example 6 of the paper).
    if "education" in df.columns:
        education_tiers = {
            "Preschool": "<HS", "1st-4th": "<HS", "5th-6th": "<HS", "7th-8th": "<HS",
            "9th": "<HS", "10th": "<HS", "11th": "<HS", "12th": "<HS",
            "HS-grad": "HS-grad",
            "Some-college": "Some-college", "Assoc-acdm": "Some-college", "Assoc-voc": "Some-college",
            "Bachelors": "Bachelors",
            "Masters": "Grad", "Prof-school": "Grad", "Doctorate": "Grad",
        }
        df["education"] = df["education"].map(education_tiers)

    sfm = build_sfm(
        sensitive_attr=config["protected"],
        outcome_attr=config["target_col"],
        confounder_attrs=config["confounders"],
        mediator_attrs=config["mediators"],
        sorted_mediators=len(config["mediators"]) > 1,
        sorted_confounders=len(config["confounders"]) > 1,
    )
    # Laplace smoothing with pseudo-count alpha = 1 on every state of every
    # variable, i.e. P(state | parents) = (count + 1) / (N_parents + n_states),
    # matching the parameter estimation described in the reference paper.
    # pgmpy's "K2" is a shorthand for exactly this; the explicit dirichlet form
    # is used here because it states alpha = 1 in the code.
    bn = fit_discrete_bayesian_model(
        sfm=sfm,
        data=df,
        estimator_instance=(
            BayesianEstimator,
            {"prior_type": "dirichlet", "pseudo_counts": 1},
        ),
    )

    target = (config["target_col"], config["target_val"])
    x0, x1 = config["x0"], config["x1"]

    start = time.perf_counter()
    tv = total_variation(bn, target, config["protected"], x0, x1)
    te = total_effect(bn, target, config["protected"], x0, x1)
    effects = {
        "TV": tv,
        "TE": te,
        # SE = TV - TE (Eq. 3, Plecko & Bareinboim 2024), the same identity
        # the prompt asks the LLM to apply.
        "SE": tv - te,
        "DE": natural_direct_effect(bn, target, config["protected"], x0, x1),
        # The paper defines the indirect effect twice, and the two are different
        # quantities, not a sign flip. Both are computed here.
        #
        # "IE" is IE_{x0,x1}, the identification formula of Eq. 8, which is the
        # one written in the prompt of 2_3, where the LLM is asked to compute
        # it. In this notebook nothing is asked of the LLM numerically, so it
        # is carried only for completeness.
        #
        # "IE_reverse" is IE_{x1,x0}, the form Prop. 2 (Eq. 9) puts in the
        # decomposition TE = DE - IE. Only this one makes that identity close,
        # so it is the value the report shows and the one the Recap Questions
        # are scored against: Q2 flips if the direct form is used instead.
        "IE": natural_indirect_effect(bn, target, config["protected"], x0, x1),
        "IE_reverse": natural_indirect_effect(bn, target, config["protected"], x1, x0),
    }
    elapsed = time.perf_counter() - start

    return effects, bn, len(df), elapsed

ground_truth, bn, n_rows, fairmind_time = run_fairmind(CONFIG)
print(f"FairMind - elapsed time: {fairmind_time:.4f}s  ({n_rows} rows)")
for k, v in ground_truth.items():
    print(f"  {k}: {v:.6f}")

# Eq. 9 closes on the reverse form only; a mismatch here means the two IE
# definitions have been mixed up again.
check = ground_truth["DE"] - ground_truth["IE_reverse"]
print(f"\n  decomposition check: DE - IE_reverse = {check:.6f}  vs  TE = {ground_truth['TE']:.6f}")

## 5. Building the prompts

`build_prompts()` injects the exact values and the metadata into the LaTeX template **in Python**, before the model sees it: the LLM never receives an empty numeric placeholder and has no way of altering the figures. Only the eight `<<...>>` placeholders are left to fill, three qualitative blocks and the five recap answers.

The table row for the indirect effect carries the reverse form, since the document presents a decomposition. The full dictionary, with both forms, is the one passed to the validator and saved to disk.

Column names go through `escape_latex()`: without it an underscore, as in `S2_gender`, makes the LaTeX compilation fail with `Missing $ inserted`.

In [ ]:
import datetime

REPORT_DATE = datetime.date.today().isoformat()

context = {
    "dataset": CONFIG["dataset_name"],
    "protected_attr": CONFIG["protected"],
    "x0": CONFIG["x0"],
    "x1": CONFIG["x1"],
    "outcome_attr": f"{CONFIG['target_col']} ({CONFIG['target_val']})",
    "mediator": ", ".join(CONFIG["mediators"]),
    "confounder": ", ".join(CONFIG["confounders"]),
}

# The document presents a decomposition, so its "IE" row must carry the
# reverse form. The full dictionary, with both forms, is what goes to the
# validator and to disk; only the report view is remapped.
report_effects = {**ground_truth, "IE": ground_truth["IE_reverse"]}

system_prompt, user_prompt = build_prompts(report_effects, context, REPORT_DATE)

print(f"system_prompt: {len(system_prompt)} caratteri")
print(f"user_prompt:   {len(user_prompt)} caratteri")
print()
print("Placeholder left to LLM:",
      find_unfilled_placeholders(user_prompt))

## 6. Calling the LLM

The model receives the prompts and returns the complete LaTeX document. We use `call_llm_report()` rather than `src.llm.call_llm()`: the latter looks for a JSON block in the response and would fail here, since it is meant for the effects benchmark.

`cache_prompt=False` is passed explicitly for the reason documented in `2_3`: with prompt caching left on, identical requests may diverge even at temperature 0, and a report that is not reproducible cannot be evaluated.

`DRY_RUN = True` exercises the remaining cells locally, without the server on Thor, by filling the placeholders with synthetic text and fixed answers. It does not produce a valid result: it only checks that parser, compilation and scoring work end to end, and it is recorded in the final JSON so that a rehearsal can never be mistaken for a run.

In [ ]:
DRY_RUN = False   # True = no network call, simulated output (NOT a valid result)

if DRY_RUN:
    import time as _time
    _mock = extract_latex_document(user_prompt)
    for _ph in ["QUALITATIVE_TOTAL", "QUALITATIVE_DE", "QUALITATIVE_IE"]:
        _mock = _mock.replace(f"<<{_ph}>>", "[SIMULATED TEXT - DRY RUN, not real content]")
    for _i in range(1, 6):
        _mock = _mock.replace(f"<<ANSWER_Q{_i}>>", "SI")
    latex_report = _mock
    llm_usage = {"input_tokens": None, "output_tokens": None,
                 "total_tokens": None, "finish_reason": "dry_run"}
    llm_time = 0.0
    print("DRY RUN - no call to the LLM was made.")
else:
    # cache_prompt=False: see the note in the markdown above and in 2_3.
    latex_report, llm_usage, llm_time = call_llm_report(
        system_prompt, user_prompt, max_tokens=4096, cache_prompt=False,
    )
    print(f"LLM - time: {llm_time:.2f}s")
    print(f"Token: input={llm_usage['input_tokens']}, "
          f"output={llm_usage['output_tokens']}, "
          f"total={llm_usage['total_tokens']} "
          f"(finish_reason={llm_usage['finish_reason']})")

leftover = find_unfilled_placeholders(latex_report)
if leftover:
    print(f"\nATTENZIONE: segnaposto non riempiti dall'LLM: {leftover}")
else:
    print("\nAll placeholders were successfully filled.")

print(f"Document length: {len(latex_report)} characters")

## 7. Saving the `.tex` and checking that it compiles

Saves the report and, when `pdflatex` is available, compiles it in a temporary directory. A report that does not compile is a failure of the pipeline even if every answer is correct, which is why the check is automated rather than left to inspection.

In [ ]:
import os
import shutil
import subprocess
import tempfile

os.makedirs("benchmark_results/reports", exist_ok=True)
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
tex_path = f"benchmark_results/reports/{CONFIG['dataset_name']}_{ts}.tex"

with open(tex_path, "w", encoding="utf-8") as f:
    f.write(latex_report)
print(f"Report saved: {tex_path}")

compiles = None
if shutil.which("pdflatex") is None:
    print("pdflatex not available: compilation check skipped.")
else:
    with tempfile.TemporaryDirectory() as tmp:
        shutil.copy(tex_path, os.path.join(tmp, "report.tex"))
        proc = subprocess.run(
            ["pdflatex", "-interaction=nonstopmode", "-halt-on-error", "report.tex"],
            cwd=tmp, capture_output=True, text=True,
        )
        compiles = proc.returncode == 0
    if compiles:
        print("LaTeX compilation: OK")
    else:
        print("LaTeX compilation: FAILED")
        print("\n".join(l for l in proc.stdout.splitlines() if l.startswith("!"))[:500])

## 8. Scoring the Recap Questions

`score_report()` extracts the five yes/no answers from the document and compares them with a ground truth computed **from the FairMind numbers alone**, through deterministic threshold rules. A missing or unreadable answer counts as wrong: a report that does not respect the format cannot score full marks.

The rules for Q2 and Q5 concern the decomposition and therefore read `IE_reverse`. The validator raises an error if that key is absent, rather than falling back on the direct form, because such a fallback would produce a wrong ground truth with no observable symptom.

In [ ]:
print("Thresholds used for the ground truth:")
print(f"  THRESH_DE     = {THRESH_DE}    (|DE| above => direct discrimination)")
print(f"  EPS_IE        = {EPS_IE}   (|IE| below => negligible channel)")
print(f"  THRESH_TV     = {THRESH_TV}    (|TV| above => practical relevance)")
print(f"  THRESH_SE_REL = {THRESH_SE_REL}    (|SE|/|TV| above => substantial spurious part)")
print()

score = score_report(latex_report, ground_truth)
result = score.to_dict()

questions_df = pd.DataFrame(result["questions"])
print(questions_df.to_string(index=False))
print()
print(f"CONSISTENCY SCORE: {result['n_correct']}/{result['n_total']} "
      f"= {result['score_pct']}")
if result["n_unparseable"]:
    print(f"({result['n_unparseable']} unreadable answers)")

# The expected answers cannot appear in the template the model receives: it
# would read and copy them, and the score would stop measuring anything. They
# are therefore added here, to the document already generated and scored, so
# that the two columns can be compared side by side. The original file stays
# on disk as evidence of what the model actually wrote.
annotated_report = annotate_recap_answers(latex_report, score)
annotated_path = tex_path.replace(".tex", "_annotated.tex")
with open(annotated_path, "w", encoding="utf-8") as f:
    f.write(annotated_report)
print(f"\nAnnotated report (LLM answer vs expected): {annotated_path}")

## 9. Saving the results

Writes a JSON next to the `.tex` with the FairMind effects, the score, the outcome of the compilation and the token and timing metrics. The `dry_run` flag distinguishes an offline rehearsal from a real execution beyond any doubt.

In [ ]:
out = {
    "dataset": CONFIG["dataset_name"],
    "timestamp": ts,
    "dry_run": DRY_RUN,
    "config": {k: v for k, v in CONFIG.items() if k != "csv_path"},
    "fairmind_effects": ground_truth,
    # Reso esplicito: le regole di Q2 e Q5 leggono la forma inversa.
    "ie_form_used_for_scoring": "IE_reverse",
    "decomposition_check": {
        "DE_minus_IE_reverse": round(ground_truth["DE"] - ground_truth["IE_reverse"], 6),
        "TE": round(ground_truth["TE"], 6),
    },
    "report_tex": tex_path,
    "report_tex_annotated": annotated_path,
    "latex_compiles": compiles,
    "unfilled_placeholders": leftover,
    "scoring": result,
    "thresholds": {
        "THRESH_DE": THRESH_DE,
        "EPS_IE": EPS_IE,
        "THRESH_TV": THRESH_TV,
        "THRESH_SE_REL": THRESH_SE_REL,
    },
    "token_usage": llm_usage,
    "timing": {
        "fairmind_seconds": round(fairmind_time, 4),
        "llm_seconds": round(llm_time, 4),
    },
}

json_path = f"benchmark_results/reports/{CONFIG['dataset_name']}_{ts}.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False)
print(f"Results saved: {json_path}")